# Experiment: Offline MCMC-IS Objective Grid Study

Objective:
- Evaluate a deterministic, production-like discrete hyperparameter grid for MCMC-IS.
- Compare realistic production objectives against oracle RMSE on the same scenarios used in the production notebooks.
- Quantify objective noise across repeat seeds without running the old long final-budget follow-up.

In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    DEFAULT_MCMC_OBJECTIVE_GRID_Q_MULTIPLIERS,
    DEFAULT_MCMC_OBJECTIVE_GRID_SWAP_COUNTS,
    MCMCWorkflowConfig,
    MCMC_OBJECTIVE_GRID_REALISTIC_OBJECTIVES,
    SAMCWorkflowConfig,
    _mcmc_eval_count,
    _steps_per_chain,
    build_beta_initialization,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_cross_method_saved_output,
    load_beta_sweep_saved_output,
    load_mcmc_objective_grid_saved_output,
    load_selected_scenarios,
    plot_named_method_convergence,
    plot_named_method_max_budget,
    read_json,
    run_named_mcmc_checkpoint_study,
    run_mcmc_objective_grid_study,
    save_mcmc_objective_grid_outputs,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
    summarize_records,
    tune_samc_setup,
    write_json,
    write_jsonl,
    _effective_n_jobs,
    _iid_replicate_worker,
    _samc_replicate_worker,
    _try_make_process_pool,
)

pd.set_option("display.max_columns", 100)
project_root

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Configuration

This notebook is intentionally heavy by default.  
It is an offline objective study, not a long-run estimator comparison.

In [ ]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "mcmcis_objective_grid"

SCENARIO_GROUP = "exploratory_exact"
SCENARIO_KEYS_OVERRIDE = None

Q_MULTIPLIERS = DEFAULT_MCMC_OBJECTIVE_GRID_Q_MULTIPLIERS
N_SWAP_PAIRS = DEFAULT_MCMC_OBJECTIVE_GRID_SWAP_COUNTS
TRIAL_REPEATS = 5 if not FAST_MODE else 2
TRIAL_BUDGET = 200_000 if not FAST_MODE else 20_000
EXTRA_TRIAL_BUDGETS = tuple()
Q_FLOOR = 1e-12
N_JOBS = min(os.cpu_count() or 1, len(Q_MULTIPLIERS) * len(N_SWAP_PAIRS) * TRIAL_REPEATS)
MIN_TAIL_STATES = 2
BASE_SEED = 31_415

mcmc_cfg = MCMCWorkflowConfig(
    pilot_samples=20_000 if not FAST_MODE else 1_000,
    scale_method="sd",
    beta_max_init=1e6,
    chains=2,
    burn_in_fraction=0.20,
    thin=1,
    estimate_variance=True,
    obm_batch_size=None,
    tilt_mode="smooth_hinge",
    proposal_size=0.075,
)

NOTEBOOK_CONFIG = {
    "FAST_MODE": FAST_MODE,
    "SCENARIO_GROUP": SCENARIO_GROUP,
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "Q_MULTIPLIERS": Q_MULTIPLIERS,
    "N_SWAP_PAIRS": N_SWAP_PAIRS,
    "TRIAL_REPEATS": TRIAL_REPEATS,
    "TRIAL_BUDGET": TRIAL_BUDGET,
    "EXTRA_TRIAL_BUDGETS": EXTRA_TRIAL_BUDGETS,
    "Q_FLOOR": Q_FLOOR,
    "N_JOBS": N_JOBS,
    "BASE_SEED": BASE_SEED,
}

print(json.dumps(NOTEBOOK_CONFIG, indent=2))

## Notebook Helpers

In [ ]:
REALISTIC_OBJECTIVES = list(MCMC_OBJECTIVE_GRID_REALISTIC_OBJECTIVES)


def plot_oracle_rmse_heatmap(
    config_summary: list[dict],
    q_multipliers: tuple[float, ...],
    n_swap_pairs_values: tuple[int, ...],
    scenario_name: str,
    *,
    save_path: Path | None = None,
) -> None:
    df = pd.DataFrame(config_summary)
    heat = np.full((len(n_swap_pairs_values), len(q_multipliers)), np.nan, dtype=float)
    for _, row in df.iterrows():
        swap_idx = list(n_swap_pairs_values).index(int(row["n_swap_pairs"]))
        q_idx = int(row["q_index"])
        val = float(row["mean_oracle_rmse"])
        heat[swap_idx, q_idx] = np.log10(val) if np.isfinite(val) and val > 0.0 else np.nan

    fig, ax = plt.subplots(figsize=(13, 4.5))
    im = ax.imshow(heat, aspect="auto", cmap="viridis")
    ax.set_title(f"Oracle RMSE grid: {scenario_name}")
    ax.set_xlabel("q_multiplier")
    ax.set_ylabel("n_swap_pairs")
    ax.set_xticks(np.arange(len(q_multipliers)))
    ax.set_xticklabels([f"{v:g}" for v in q_multipliers], rotation=45, ha="right")
    ax.set_yticks(np.arange(len(n_swap_pairs_values)))
    ax.set_yticklabels([str(v) for v in n_swap_pairs_values])
    fig.colorbar(im, ax=ax, label="log10(mean oracle RMSE)")
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=170, bbox_inches="tight")
    plt.close(fig)


def plot_objective_seed_noise_heatmap(
    seed_noise_rows: list[dict],
    scenario_name: str,
    *,
    save_path: Path | None = None,
) -> None:
    df = pd.DataFrame(seed_noise_rows)
    metrics = ["exact_match_rate", "mean_fuzzy_similarity"]
    heat = np.asarray([[float(df.loc[df["objective_name"] == obj, metric].iloc[0]) for metric in metrics] for obj in REALISTIC_OBJECTIVES], dtype=float)
    fig, ax = plt.subplots(figsize=(8, max(4.5, 0.35 * len(REALISTIC_OBJECTIVES))))
    im = ax.imshow(heat, aspect="auto", cmap="magma", vmin=0.0, vmax=1.0)
    ax.set_title(f"Objective seed-noise summary: {scenario_name}")
    ax.set_xticks(np.arange(len(metrics)))
    ax.set_xticklabels(metrics, rotation=20, ha="right")
    ax.set_yticks(np.arange(len(REALISTIC_OBJECTIVES)))
    ax.set_yticklabels(REALISTIC_OBJECTIVES)
    fig.colorbar(im, ax=ax, label="score")
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=170, bbox_inches="tight")
    plt.close(fig)


def build_cross_scenario_leaderboard(objective_rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame([row for row in objective_rows if row["objective_kind"] == "realistic"])
    leaderboard = (
        df.groupby("objective_name", as_index=False)
        .agg(
            exact_match_count=("oracle_exact_match", "sum"),
            mean_fuzzy_similarity=("oracle_fuzzy_similarity", "mean"),
            mean_q_index_distance=("oracle_q_index_distance", "mean"),
            mean_swap_distance=("oracle_swap_distance", "mean"),
        )
        .sort_values(
            ["exact_match_count", "mean_fuzzy_similarity", "mean_q_index_distance", "mean_swap_distance"],
            ascending=[False, False, True, True],
        )
        .reset_index(drop=True)
    )
    return leaderboard


def plot_cross_scenario_fuzzy_similarity(
    objective_rows: list[dict],
    *,
    save_path: Path | None = None,
) -> None:
    df = pd.DataFrame([row for row in objective_rows if row["objective_kind"] == "realistic"])
    scenarios = list(dict.fromkeys(df["scenario_key"]))
    heat = np.full((len(REALISTIC_OBJECTIVES), len(scenarios)), np.nan, dtype=float)
    for _, row in df.iterrows():
        obj_idx = REALISTIC_OBJECTIVES.index(str(row["objective_name"]))
        scn_idx = scenarios.index(str(row["scenario_key"]))
        heat[obj_idx, scn_idx] = float(row["oracle_fuzzy_similarity"])

    fig, ax = plt.subplots(figsize=(10, max(4.5, 0.35 * len(REALISTIC_OBJECTIVES))))
    im = ax.imshow(heat, aspect="auto", cmap="viridis", vmin=0.0, vmax=1.0)
    ax.set_title("Cross-scenario fuzzy similarity to oracle RMSE")
    ax.set_xticks(np.arange(len(scenarios)))
    ax.set_xticklabels(scenarios, rotation=25, ha="right")
    ax.set_yticks(np.arange(len(REALISTIC_OBJECTIVES)))
    ax.set_yticklabels(REALISTIC_OBJECTIVES)
    fig.colorbar(im, ax=ax, label="fuzzy similarity")
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=170, bbox_inches="tight")
    plt.close(fig)

## Load Scenarios

In [ ]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_OVERRIDE,
    portfolio_group=None if SCENARIO_KEYS_OVERRIDE is not None else SCENARIO_GROUP,
    min_tail_states=MIN_TAIL_STATES,
)

run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "objective_grid") if SAVE_OUTPUTS else None

pd.DataFrame([
    {
        "scenario": s.key,
        "exact_p": s.exact_p,
        "tail_hits": s.exact_tail_hits,
        "n_perm": s.exact_n_perm,
        "exact_method": s.exact_method,
        "family": s.portfolio.get("family"),
        "rarity_band": s.portfolio.get("rarity_band"),
        "difficulty": s.portfolio.get("expected_difficulty"),
        "groups": ",".join(s.portfolio.get("groups", [])),
    }
    for s in scenarios
])

## Run Offline Objective Grid Study

In [ ]:
grid_results = {}
cross_scenario_objective_rows = []

for scenario_idx, scenario in enumerate(scenarios):
    print(f"Running offline objective grid for {scenario.key} | exact p={scenario.exact_p:.3e}")
    study_seed = BASE_SEED + 10_000 * (scenario_idx + 1)
    grid_study = run_mcmc_objective_grid_study(
        scenario.problem,
        scenario.exact_p,
        mcmc_cfg=mcmc_cfg,
        trial_repeats=TRIAL_REPEATS,
        trial_budget=TRIAL_BUDGET,
        base_seed=study_seed,
        q_multipliers=Q_MULTIPLIERS,
        n_swap_pairs_values=N_SWAP_PAIRS,
        q_floor=Q_FLOOR,
        n_jobs=N_JOBS,
    )

    objective_winners_df = pd.DataFrame(grid_study["objective_winners"]).sort_values(["objective_kind", "objective_name"])
    objective_winners_df["scenario_key"] = scenario.key
    objective_winners_df["family"] = scenario.portfolio.get("family")
    objective_winners_df["rarity_band"] = scenario.portfolio.get("rarity_band")
    cross_scenario_objective_rows.extend(objective_winners_df.to_dict(orient="records"))

    scenario_dir = (run_dir / scenario.key) if (SAVE_OUTPUTS and run_dir is not None) else None
    if scenario_dir is not None:
        plot_oracle_rmse_heatmap(
            grid_study["config_summary"],
            Q_MULTIPLIERS,
            N_SWAP_PAIRS,
            scenario.description,
            save_path=scenario_dir / "oracle_rmse_heatmap.png",
        )
        plot_objective_seed_noise_heatmap(
            grid_study["objective_seed_noise"],
            scenario.description,
            save_path=scenario_dir / "objective_seed_noise_heatmap.png",
        )
        save_mcmc_objective_grid_outputs(
            grid_study,
            output_dir=scenario_dir,
            scenario_name=scenario.description,
            exact_p=scenario.exact_p,
            notebook_config=NOTEBOOK_CONFIG,
        )

    grid_results[scenario.key] = {
        "grid_study": grid_study,
    }

    print(json.dumps({
        "scenario": scenario.key,
        "sigma_t": grid_study["study_context"]["sigma_t"],
        "q_target": grid_study["study_context"]["q_target"],
        "oracle_winner": grid_study["oracle_winner"],
    }, indent=2))

    display(pd.DataFrame(grid_study["config_summary"]).sort_values("mean_oracle_rmse").head(12)[[
        "config_id",
        "beta",
        "n_swap_pairs",
        "q_multiplier",
        "q_trial",
        "mean_oracle_rmse",
        "mean_oracle_abs_log10",
        "mean_objective_varhat",
        "mean_objective_varhat_qmatch_soft",
        "mean_objective_varhat_degeneracy_soft",
        "mean_objective_varhat_qmatch_degeneracy_soft",
        "mean_tail_hits",
        "mean_acceptance_rate",
        "mean_ess",
        "mean_weight_cv",
    ]])
    display(objective_winners_df[[
        "objective_name",
        "config_id",
        "q_multiplier",
        "n_swap_pairs",
        "beta",
        "selected_objective_value",
        "oracle_exact_match",
        "oracle_fuzzy_similarity",
        "oracle_q_index_distance",
        "oracle_swap_distance",
    ]])
    display(pd.DataFrame(grid_study["objective_seed_noise"]).sort_values("objective_name"))

## Cross-Scenario Objective Summary

In [ ]:
cross_scenario_df = pd.DataFrame(cross_scenario_objective_rows)
realistic_cross_scenario_df = cross_scenario_df[cross_scenario_df["objective_kind"] == "realistic"].copy()
leaderboard_df = build_cross_scenario_leaderboard(cross_scenario_objective_rows)
display(leaderboard_df)
display(
    realistic_cross_scenario_df.groupby(["family", "rarity_band", "objective_name"], as_index=False)
    .agg(
        mean_fuzzy_similarity=("oracle_fuzzy_similarity", "mean"),
        exact_match_count=("oracle_exact_match", "sum"),
    )
    .sort_values(["family", "rarity_band", "objective_name"])
)
display(realistic_cross_scenario_df.sort_values(["objective_name", "scenario_key"])[[
    "scenario_key",
    "family",
    "rarity_band",
    "objective_name",
    "config_id",
    "q_multiplier",
    "n_swap_pairs",
    "beta",
    "oracle_exact_match",
    "oracle_fuzzy_similarity",
    "oracle_q_index_distance",
    "oracle_swap_distance",
]])

if SAVE_OUTPUTS and run_dir is not None:
    plot_cross_scenario_fuzzy_similarity(
        cross_scenario_objective_rows,
        save_path=run_dir / "cross_scenario_fuzzy_similarity_heatmap.png",
    )
    display(Image(filename=str(run_dir / "cross_scenario_fuzzy_similarity_heatmap.png")))
else:
    plot_cross_scenario_fuzzy_similarity(cross_scenario_objective_rows)
    print("SAVE_OUTPUTS=False, so the cross-scenario heatmap was not saved.")

## Review Saved Figures

In [ ]:
if SAVE_OUTPUTS and run_dir is not None:
    print(f"Saved outputs under: {run_dir}")
    for scenario in scenarios:
        scenario_dir = run_dir / scenario.key
        print(f"\n{scenario.key}")
        display(Image(filename=str(scenario_dir / "oracle_rmse_heatmap.png")))
        display(Image(filename=str(scenario_dir / "objective_seed_noise_heatmap.png")))
else:
    print("SAVE_OUTPUTS=False, so no saved figures to display.")

## Reload Saved Results Without Rerunning

In [ ]:
# RELOAD_GRID_DIR = None
# # Example:
# # RELOAD_GRID_DIR = project_root / "results" / "mcmcis_objective_grid" / "20260312_120000_objective_grid" / "linear_stat_dp_n40"

# if RELOAD_GRID_DIR is not None:
#     saved = load_mcmc_objective_grid_saved_output(RELOAD_GRID_DIR)
#     print(json.dumps({
#         "scenario_display": saved["config_summary_payload"]["scenario_display"],
#         "exact_p": saved["config_summary_payload"]["exact_p"],
#     }, indent=2))
#     display(pd.DataFrame(saved["config_summary_payload"]["config_summary"]).head())
#     display(pd.DataFrame(saved["objective_winners_payload"]["objective_winners"]).head())
#     display(pd.DataFrame(saved["objective_seed_noise_payload"]["objective_seed_noise"]).head())
# else:
#     print("Set RELOAD_GRID_DIR to a saved objective-grid scenario directory to inspect saved results.")

## Optional Lower-Budget Selector Stress Test

In [ ]:
# Set EXTRA_TRIAL_BUDGETS above to something like (100_000, 50_000) to rerun the
# same objective grid at smaller scan budgets and compare whether the realistic
# objective ranking remains stable.
if EXTRA_TRIAL_BUDGETS:
    budget_leaderboards = {}
    for extra_budget in EXTRA_TRIAL_BUDGETS:
        rows = []
        for scenario_idx, scenario in enumerate(scenarios):
            study_seed = BASE_SEED + 200_000 + 10_000 * (scenario_idx + 1) + int(extra_budget)
            rerun = run_mcmc_objective_grid_study(
                scenario.problem,
                scenario.exact_p,
                mcmc_cfg=mcmc_cfg,
                trial_repeats=TRIAL_REPEATS,
                trial_budget=int(extra_budget),
                base_seed=study_seed,
                q_multipliers=Q_MULTIPLIERS,
                n_swap_pairs_values=N_SWAP_PAIRS,
                q_floor=Q_FLOOR,
                n_jobs=N_JOBS,
            )
            df = pd.DataFrame(rerun["objective_winners"])
            df["scenario_key"] = scenario.key
            rows.extend(df.to_dict(orient="records"))
        budget_leaderboards[int(extra_budget)] = build_cross_scenario_leaderboard(rows)

    for budget, leaderboard in budget_leaderboards.items():
        print(f"\nTrial budget = {budget:,}")
        display(leaderboard)
else:
    print("Set EXTRA_TRIAL_BUDGETS to rerun the objective grid at smaller scan budgets.")